# Case set up
### Import libraries

In [1]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib as jl
threads = 4

from tqdm import tqdm
from sklearn.model_selection import train_test_split

# tf.config.threading.set_inter_op_parallelism_threads(1)
# tf.config.threading.set_intra_op_parallelism_threads(1)

Visual set up

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib qt5

# Use custom style
style_name = 'paper'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] + '/RESEARCH/styles/matplotlib/' + style_name + '.mplstyle'
) # See: https://matplotlib.org/1.5.3/users/style_sheets.html

In [3]:
# Get colors list
colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# Figure/animation saving parameters
# mpl.rcParams['savefig.dpi'] = 100

# Font parameters
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# mpl.rcParams['text.latex.preamble'] = join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Define PrODE and case (model) directories paths

In [4]:
prode_dir = os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/'
case_dir = "./trained/delta/"

In [5]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(os.path.normpath(prode_dir))
sys.path.append(os.path.normpath(case_dir))
cwd = os.getcwd()

### Import additional libraries

In [6]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
# Physics-related
from physics.data_generator import DataGenerator
from prode.physics.general import const
from prode.physics.thermochemistry import specie as specieMdl
from prode.physics.thermochemistry.reactionThermo import mixture as mixMdl

Change directory

In [7]:
os.chdir(os.path.normpath(case_dir))

### Initialize network

In [8]:
net = fun._net(
    input_module=inputs
)


[PrODE]: Initializing 'NeqCtrlNN' network ...


### Initialize PrODE model

In [9]:
from inputs.general import flags

flags['load'] = 1

for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)


[PrODE]: Building 'NeqCtrlNN' network ...
Model: "NonEqLevel_MMOpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 29)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 1)]          0                                            
__________________________________________________________________________________________________
branch_fn (Lambda)              (None, 29)           0           input_1[0][0]                    
__________________________________________________________________________________________________
trunk_fn (Scaling)              (None, 1)            2           input_2[0][0]                    
_______________________________________

### Initialize testing data generator
Choose specific test scenarios

In [13]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = case_dir + "/data/"
# if not os.path.exists(dataGen.data_dir):
#     os.makedirs(dataGen.data_dir)

# utils.print_main("Generating testing pts ...")
# if inputs.data.from_file:
#     test_data = dataGen.testing_data(
#         inputs.data.file['test'],
#         header=inputs.data.header,
#         indices=indices-1
#     )
# else:
#     raise NotImplementedError

  0%|          | 0/3 [00:00<?, ?it/s]


[PrODE]: Generating testing pts ...


100%|██████████| 3/3 [00:02<00:00,  1.19it/s]


Change directory

In [ ]:
os.chdir(os.path.normpath(cwd))

# Postprocessing - Model analysis

## Set up analysis

In [ ]:
path = cwd + '/outputs/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'pred': 1,
    'input_space': 0,
    'one_pass': 0,
    'accuracy': 0,
    'comp_time': 1,
    'speedup': 0,
    'recursive': 0,
    'noneq_dist': 0,
    'levels': 0,
    'animations': 0,
    'adaptive': 0,
    'park_ts1': 0
}

#### Get predictions

In [ ]:
if FLAGS['pred']:
    utils.print_main("Postprocessing physical quantities ...")
    x, y, y_pred = [], [], []
    for scenario in tqdm(test_data):
        x_i, y_i = scenario
        x.append(x_i)
        y.append(y_i)
        y_pred.append(model.net.call(x_i))
    N = len(test_data)

## Adaptive technique
Setting up

In [326]:
from src.chemistryModel import ChemistryModel
from src.adaptive import AdaptiveSolver
from src.noneqController import NoneqController

if FLAGS['adaptive']:
    path_ = path + '/adaptive/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Define noneq. metrics tolerances
tol_list = [0.01, 0.1, 0.5]

Analisys of grouped-reconstructed rate coefficients
\begin{align*}
    \frac{dn_P}{dt}&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Multiply by }\mathcal{F}_p\mathcal{F}_q:\\
    \mathcal{F}_p\mathcal{F}_q\frac{dn_P}{dt}&=
    \mathcal{F}_p\mathcal{F}_q\left(-\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3\right)
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Since }n_p&=n_P\mathcal{F}_p\text{ and }n_q=n_Q\mathcal{F}_q:\\
    \frac{dn_p}{dt}\mathcal{F}_q&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    %&&\text{Sum over }\mathcal{I}_q:\\
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\frac{dn_p}{dt}\mathcal{F}_q&=
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{PQ}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}n_pn_\text{O}+K^R_{P}\mathcal{F}_pn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{pq}n_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{qp}n_qn_\text{O}
    -\bar{K}^D_{p}n_pn_\text{O}+\bar{K}^R_{p}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
\end{align*}
So
\begin{align*}
    \bar{K}_{pq}&=K_{PQ}\mathcal{F}_q\\
    \bar{K}_{qp}&=K_{QP}\mathcal{F}_p\\
    \bar{K}^D_{p}&=K_P^D\\
    \bar{K}^R_{p}&=K_P^R\mathcal{F}_p
\end{align*}

Initialize chemistry models and adaptive solver

In [327]:
chemModels = []
for i, spThs in enumerate(speciesThermos):
    pf = copy.deepcopy(inputs.general.processFlags)
    if i == 0:
        pf['excit'] = 0
    chemModels.append(
        ChemistryModel(
            specieThermos=spThs,
            processFlags=pf
        )
    )
    
adaptiveSolver = AdaptiveSolver(
    noneqController=noneqController,
    chemModels=chemModels
)
    
# adaptiveSolver = AdaptiveSolver(
#     model=model,
#     noneqController=noneqController,
#     chemModels=chemModels
# )

[PrODE]:   WARNING! No specific `reactionsList` have been provided to the chemistry model.
[PrODE]:   >> The following one will be taken:
[PrODE]:   >> `/home/zanardi/WORKSPACE/ML/PrODE/prode/prode/physics/database/air/kinetics/O3_UMN/RVE1/reactions.csv`
[PrODE]:   WARNING! No specific `reactionsList` have been provided to the chemistry model.
[PrODE]:   >> The following one will be taken:
[PrODE]:   >> `/home/zanardi/WORKSPACE/ML/PrODE/prode/prode/physics/database/air/kinetics/O3_UMN/RVE3/reactions.csv`
[PrODE]:   WARNING! No specific `reactionsList` have been provided to the chemistry model.
[PrODE]:   >> The following one will be taken:
[PrODE]:   >> `/home/zanardi/WORKSPACE/ML/PrODE/prode/prode/physics/database/air/kinetics/O3_UMN/RVE9/reactions.csv`
[PrODE]:   WARNING! No specific `reactionsList` have been provided to the chemistry model.
[PrODE]:   >> The following one will be taken:
[PrODE]:   >> `/home/zanardi/WORKSPACE/ML/PrODE/prode/prode/physics/database/air/kinetics/O3_UMN/

#### Custom ICs

In [328]:
# Ttr = [14000., 8000.]
# T0 = [1000., 7500.]
# P0 = 8.e3
# XO2 = 0.95

# inp = {
#     'noneq': [Ttr[0], T0[0], P0, XO2],
#     'eq': [Ttr[1], T0[1], P0, XO2]
# }

Ttr = [12200., 8000.]
T0 = [2900., 8000.]
P0 = [4.6e3, 7.e3]
XO2 = [0.38, 0.6]

# Ttr = [12000., 8000.]
# T0 = [3000., 8000.]
# P0 = [4.5e3, 7.e3]
# XO2 = [0.4, 0.6]

inp = {
    'noneq': [Ttr[0], T0[0], P0[0], XO2[0]],
    'eq': [Ttr[1], T0[1], P0[1], XO2[1]]
}

xlim = [1.e-12, 1.e-3]


In [329]:
if FLAGS['adaptive']:
    # Generating Initial Conditions
    rho_s = np.zeros((2,dataGen.num_bins+1), dtype=np.float64)
    for i, inp_i in enumerate(inp.values()):
        Ttr, T0, P0, XO2 = inp_i
        XO = 1. - XO2
        nd = P0 / (const.UKB * T0)
        Q_bins = dataGen.partition_func(T0)
        rho_s[i,:-1] = XO2 * Q_bins / sum(Q_bins) * 2.0 * const.mass['O'] * nd
        rho_s[i, -1] = XO  * const.mass['O'] * nd
    rho = np.sum(rho_s, axis=1, keepdims=True)
    Ttr = np.tile(utils.reshape_to_2DArray(Ttr), (2, 1))
    ic = np.hstack((Ttr, rho, np.divide(rho_s, rho)))

    # Evaluate exact solutions
    data_adpt = dataGen.testing_data(ic_test=ic, save=False)
    data_adpt = {k: data_adpt[i] for i, k in enumerate(inp.keys())}

    # Evaluate predicted solutions
    x_adpt, y_adpt, y_pred_adpt = [], [], []
    for k, dat in tqdm(data_adpt.items()):
        x_i, y_i = dat
        x_adpt.append(x_i)
        y_adpt.append(y_i)
        y_pred_adpt.append(model.net.predict(x_i))

100%|██████████| 2/2 [00:00<00:00,  3.54it/s]


In [330]:
# %matplotlib inline
# for k, v in data_adpt.items():
#     plt.loglog(v[0][1][1:], v[1][0][:,1:2][1:], v[0][1][1:], v[1][-1][1:])
#     plt.show()

In [331]:
# identifier = 'eq'

# data = tf.nest.map_structure(lambda x: x[1:], copy.deepcopy(custom_data[identifier]))
# data = copy.deepcopy(custom_data[identifier])

In [332]:
# for i in range(len(x_adpt)):
#     x_ = x_adpt[i][-1]
#     y_ = output_ops(y_adpt[i], -1)
#     y_pred_ = output_ops(y_pred_adpt[i], -1)
#     plot_test(
#         x_,
#         y_,
#         y_pred_,
#         ts_label['TS3'],
#         fig_name='TS4',
#         labels=[r'$t$ [s]', r'$Y$'],
#         scales=['log', 'log'],
#         xlim=[1.e-12, 1.e-3],
#         save=False
#     )
#     plt.show()

Levels dynamics and nonequilibrium distance

In [333]:
# # Evaluate noneq. distance
# t, noneq_dist = noneqController.eval_df_norm(
#     data=data
# )

In [334]:
# # Set up figure parameters ----------------------------------------------------
# mpl_animation = {
#     'xtick.labelsize': 12,
#     'ytick.labelsize': 12,
#     'axes.labelsize': 12,
#     'axes.titlesize': 12,
#     'legend.fontsize': 8,
#     'figure.titlesize': 16,
#     'figure.figsize': np.array([6.4, 4.8])*1.5
# }
# # -----------------------------------------------------------------------------

# with plt.rc_context(mpl_animation):
#     # Initialize a figure with 4 subplots
#     fig, axs = plt.subplots(2,2)
#     axs = axs.reshape(-1)
#     plt.tight_layout()

#     # Plot nonequilibrium distance dynamics (3/4 subplots)
#     neqd_vlines = [] # Vertical lines for time evolution
#     for i, neqd in enumerate(noneq_dist):
#         axs[i] = plot_noneq_dist(
#             t,
#             neqd,
#             var_names=noneq_dist_names[i],
#             labels=[r'$t$ [s]', r'$\epsilon^{({%s}-4)}$' % (i+1)],
#             title=f'Nonequilibrium distance: TS{i+1}-TS4',
#             save=False,
#             fig_ax=(fig, axs[i])
#         )[1]
#         axs[i].ticklabel_format(axis='y', style="sci", scilimits=(0,0))
#         # Initialize vertical lines
#         neqd_vlines.append(
#             axs[i].axvline(0., *axs[i].get_ylim(), color='k', lw=0.5)
#         )

#     # Initialize levels distribution dynamics lines objects (4/4 subplots)
#     axs[-1], lev_lines = init_lev_dist_lines(
#         axs[-1],
#         title='States distribution dynamics'
#     )

#     def animate(frame):
#         idx = int(frame/frames*len(t))
#         # Write time instant
#         fig.suptitle(r'$t$ = %.1e s' % t[idx], fontsize=mpl_animation['figure.titlesize'])
#         # Set vertical lines data for time evolution
#         for vl in neqd_vlines:
#             vl.set_xdata(t[idx])
#         # Loop over Timescales
#         for i, (n, g, E) in enumerate(ts_lev_dist):
#             # Loop over bins
#             x, y = [], []
#             for b in range(len(n)):
#                 n_b = tf.reshape(n[b][idx,:], (-1,1))
#                 x.append(E[b])
#                 y.append(n_b/g[b])
#             X, Y = list(map(lambda z: tf.concat(z, 0).numpy().squeeze(), [x, y]))
#             lev_lines[i].set_data(X.squeeze(), Y.squeeze())
#         # Rescale axis limits
#         axs[-1].relim()
#         axs[-1].autoscale_view(tight=True)
#         return lev_lines + neqd_vlines

# # Get animation
# anim = FuncAnimation(
#     fig,
#     animate,
#     frames=frames,
#     blit=True
# )

# # Save animation
# anim.save(path_+'lev_neq_dist.mp4', fps=10, dpi=300)

# # Display animation
# HTML(anim.to_jshtml())

#### Part 1
Reconstructed groups & groups resolution for one tolerance value

In [335]:
if FLAGS['adaptive']:

#     adaptiveSolver.noneqController.tol = tol
    noneqController.tol = tol

    for i, name in enumerate(inp.keys()):

        # Define the output folder --------------------------------------------
        path_case = path_+'/part1/case_'+name+'/'+str(float(noneqController.tol))+'/'
        if not os.path.exists(path_case):
            os.makedirs(path_case)

#         Apply adaptive technique --------------------------------------------
#         t, y_adpt_sol = adaptiveSolver.solve(
#             x=copy.deepcopy(x_adpt[i]),
#             Y=copy.deepcopy(y_adpt[i]),
#     #         Y_pred=copy.deepcopy(y_adpt[i]),
#             Y_pred=copy.deepcopy(y_pred_adpt[i])
#         )

#         # Plot Y evolution of reconstructed groups vs. exact Y --------------------
#         for y_scale in ('linear', 'log'):
#         plot_test(
#             path=path_case,
#             x=t,
#             y=output_ops(y_adpt[i], -1),
#             y_pred=y_adpt_sol,
#             fig_name='TS4',
#             labels=[r'$t$ [s]', r'$Y$'],
#             scales=['log', 'log'],
#             xlim=[1.e-12, 1.e-3]
#         )
        
        # Apply adaptive technique ------------------------------------------------
        x = copy.deepcopy(x_adpt[i])
        Y = copy.deepcopy(y_adpt[i])
        df_norm = noneqController.eval_df_norm([x, Y])[-1]
        mask = tf.nest.map_structure(lambda x: x > tol, df_norm)
        
        # Apply adaptive technique --------------------------------------------
#         t, y_adpt_sol = adaptiveSolver.solve(
#             x=copy.deepcopy(x_adpt[i]),
#             Y=copy.deepcopy(y_adpt[i])
#         )

        
        # Plot Y evolution of reconstructed groups vs. exact Y --------------------
        t = x_adpt[i][-1]
        y_adpt_sol = output_ops(
            model.net.call(x=copy.deepcopy(x_adpt[i]), neq_mask=mask), -1
        )
        plot_test(
            path=path_case,
            x=t,
            y=output_ops(y_adpt[i], -1),
            y_pred=y_adpt_sol,
            fig_name='TS4',
            labels=[r'$t$ [s]', r'$Y$'],
            scales=['log', 'log'],
            xlim=[1.e-10, 1.e-3]
        )

#         # Plot groups resolution --------------------------------------------------
#         adaptiveSolver.plot_group_res(
#             path=path_case,
#             xlim=xlim,
#             fig_name='groups'
#         )

        # Plot nonequilibrium distance dynamics -----------------------------------
        path_noneq = path_case+'/metrics/'
        if not os.path.exists(path_noneq):
            os.makedirs(path_noneq)
        for i, neqd in enumerate(df_norm):
            plot_noneq_dist(
                t,
                neqd,
                var_names=noneq_dist_names[i],
                path=path_noneq,
                xlim=xlim,
                fig_name='TS'+str(i+1)
            )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Part 2
Mass fractions and number of groups needed

In [336]:
def plot_res(
    t,
    y,
    exact=True,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None,
    max_num_var=9
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    mrk = ['s','o','^']
    style = dict(
        fillstyle='full',
        markersize=3,
        markevery=y.shape[0]//20,
        lw=1.
    )
    # Define parameters
    n = y.shape[1]
    if exact:
        n = n - 1
    # Loop over features
    for d in range(n):
        ax.plot(
            t,
            y[:,d],
            marker=mrk[d],
            label=var_names[d],
            **style
        )
    if exact:
        ax.plot(
            t,
            y[:,-1],
            c='k',
            label=var_names[-1],
            linestyle='--',
            lw=1.
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='best'
    )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax

In [337]:
if FLAGS['adaptive']:
    for i, name in enumerate(inp.keys()):
        # Define the output folder ------------------------------------------------
        path_case = path_+'/part2/case_'+name+'/'
        if not os.path.exists(path_case):
            os.makedirs(path_case)

        resolution, mass, energy = [], [], []
        for j in range(len(tol_list)):
            adaptiveSolver.noneqController.tol = tol_list[j]

            # Apply adaptive technique ------------------------------------------------
            t, y_adpt_sol = adaptiveSolver.solve(
                x=copy.deepcopy(x_adpt[i]),
                Y=copy.deepcopy(y_adpt[i]),
                Y_pred=copy.deepcopy(y_adpt[i])
#                 Y_pred=copy.deepcopy(y_pred_adpt[i])
            )

            # Resolution
            resolution.append(
                np.sum(adaptiveSolver.group_res, axis=1, keepdims=True)
            )
            # Mass
            mass.append(1.-np.take(y_adpt_sol, [0], axis=1))
            # Energy
            # Get temperature and density (both constants)
            ic = x_adpt[i][0]
            T, rho = ic[0][:2]
            # Update thermos
            th = speciesThermos[-1]['O2'].thermo
            th.update(utils.reshape_to_2DArray(T))
            # Evaluate energy
            n_i = y_adpt_sol[:,1:] * rho / th.specie.M * const.UNA
#             Y_O2 = np.sum(y_adpt_sol[:,1:], axis=1, keepdims=True)
            e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
            energy.append(np.sum(e_int, axis=1, keepdims=True))

        # Plot mass fraction evolution --------------------------------------------
        # Get exact value
        mass.append(np.take(y_adpt[i][0], [0], axis=1))
        plot_res(
            t,
            np.hstack(mass),
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list] + ['Exact'],
            fig_name='mass',
            xlim=xlim,
            path=path_case
        )

        # Plot energy evolution ---------------------------------------------------
        # Get exact value
        n_i = y_adpt[i][-1] * rho / th.specie.M * const.UNA
#         Y_O2 = np.sum(y_adpt[i][-1], axis=1, keepdims=True)
        e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
        energy.append(np.sum(e_int, axis=1, keepdims=True))
        plot_res(
            t,
            np.hstack(energy),
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list] + ['Exact'],
            fig_name='energy',
            labels=[r'$t$ [s]', r'$E$ [eV]'],
            xlim=xlim,
            path=path_case
        )

        # Plot Y evolution of reconstructed groups vs. exact Y --------------------
        plot_res(
            t,
            np.hstack(resolution),
            exact=False,
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list],
            fig_name='resolution',
            labels=[r'$t$ [s]', r'\# of groups'],
            xlim=xlim,
            path=path_case,
        )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Part 3
Computational time with and without controller

In [338]:
def plot_res(
    t,
    y,
    exact=True,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None,
    max_num_var=9
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    mrk = ['s','o','^']
    style = dict(
        fillstyle='full',
        markersize=3,
        markevery=y.shape[0]//20,
        lw=1.
    )
    # Define parameters
    n = y.shape[1]
    if exact:
        n = n - 1
    # Loop over features
    for d in range(n):
        ax.plot(
            t,
            y[:,d],
            marker=mrk[d],
            label=var_names[d],
            **style
        )
    if exact:
        ax.plot(
            t,
            y[:,-1],
            c='k',
            label=var_names[-1],
            linestyle='--',
            lw=1.
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='best'
    )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax

In [339]:
xlim = [1.e-10, 1.e-3]

if FLAGS['adaptive']:
    for i, name in enumerate(inp.keys()):
        # Define the output folder ------------------------------------------------
        path_case = path_+'/part2/case_'+name+'/'
        if not os.path.exists(path_case):
            os.makedirs(path_case)

        resolution, mass, energy = [], [], []
        for j in range(len(tol_list)):
            adaptiveSolver.noneqController.tol = tol_list[j]

            # Apply adaptive technique ------------------------------------------------
            t, y_adpt_sol = adaptiveSolver.solve(
                x=copy.deepcopy(x_adpt[i]),
                Y=copy.deepcopy(y_adpt[i]),
                Y_pred=copy.deepcopy(y_adpt[i])
#                 Y_pred=copy.deepcopy(y_pred_adpt[i])
            )

            # Resolution
            resolution.append(
                np.sum(adaptiveSolver.group_res, axis=1, keepdims=True)
            )
            # Mass
            mass.append(1.-np.take(y_adpt_sol, [0], axis=1))
            # Energy
            # Get temperature and density (both constants)
            ic = x_adpt[i][0]
            T, rho = ic[0][:2]
            # Update thermos
            th = speciesThermos[-1]['O2'].thermo
            th.update(utils.reshape_to_2DArray(T))
            # Evaluate energy
            n_i = y_adpt_sol[:,1:] * rho / th.specie.M * const.UNA
#             Y_O2 = np.sum(y_adpt_sol[:,1:], axis=1, keepdims=True)
            e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
            energy.append(np.sum(e_int, axis=1, keepdims=True))

        # Plot mass fraction evolution --------------------------------------------
        # Get exact value
        mass.append(np.take(y_adpt[i][0], [0], axis=1))
        plot_res(
            t,
            np.hstack(mass),
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list] + ['Exact'],
            fig_name='mass',
            xlim=xlim,
            path=path_case
        )

        # Plot energy evolution ---------------------------------------------------
        # Get exact value
        n_i = y_adpt[i][-1] * rho / th.specie.M * const.UNA
#         Y_O2 = np.sum(y_adpt[i][-1], axis=1, keepdims=True)
        e_int = th.e_int/const.EV_to_J*th.specie.m*n_i/np.sum(n_i, axis=1, keepdims=True)
        energy.append(np.sum(e_int, axis=1, keepdims=True))
        plot_res(
            t,
            np.hstack(energy),
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list] + ['Exact'],
            fig_name='energy',
            labels=[r'$t$ [s]', r'$E$ [eV]'],
            xlim=xlim,
            path=path_case
        )

        # Plot Y evolution of reconstructed groups vs. exact Y --------------------
        plot_res(
            t,
            np.hstack(resolution),
            exact=False,
            var_names=[r'$\delta_{\text{tol}}={%s}$' % (t) for t in tol_list],
            fig_name='resolution',
            labels=[r'$t$ [s]', r'\# of groups'],
            xlim=xlim,
            path=path_case,
        )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [340]:
# identifier = 'noneq'

# # data = tf.nest.map_structure(lambda x: x[1:], copy.deepcopy(custom_data[identifier]))
# data = copy.deepcopy(custom_data[identifier])

In [341]:
# adaptiveSolver.atol = 1.e-15
# noneqController.tol = 1.e-3

In [342]:
# data = copy.deepcopy(custom_data[identifier])
# data_adapt = adaptiveSolver.solve(data)
# print(tf.nest.map_structure(lambda x: x.shape, data_adapt))

In [343]:
# data = copy.deepcopy(custom_data[identifier])
# x = data_adapt[0][1:]
# y = output_ops(data[1], -1)[1:]
# y_pred = data_adapt[1][1:]
# y_pred = np.hstack([np.take(y_pred, [-1], axis=1), np.delete(y_pred, -1, 1)])

# path_case = path_+'/case_'+identifier+'/'+str(float(noneqController.tol))+'/'
# if not os.path.exists(path_case):
#     os.makedirs(path_case)
                                              
# for y_scale in ('linear', 'log'):
#     plot_test(
#         path=path_case,
#         x=x,
#         y=y,
#         y_pred=y_pred,
#         var_names=ts_label['TS3'],
#         fig_name='TS4',
#         labels=[r'$t$ [s]', r'$Y$'],
#         scales=['log', y_scale],
#         xlim=[1.e-12, 1.e-3]
#     )

In [344]:
# adaptiveSolver.plot_group_res(
#     path=path_case,
#     xlim=[1.e-12, 1.e-3],
#     var_names=grouping_list,
#     fig_name='groups'
# )

In [345]:
# # Get solution
# t, noneq_dist = noneqController.eval_df_norm(
#     data=copy.deepcopy(custom_data[identifier])
# )
# # Plot nonequilibrium distance dynamics
# path_noneq = path_case+'/metrics/'
# if not os.path.exists(path_noneq):
#     os.makedirs(path_noneq)
# for i, neqd in enumerate(noneq_dist):
#     plot_noneq_dist(
#         t,
#         neqd,
#         var_names=noneq_dist_names[i],
#         path=path_noneq,
#         xlim=[1.e-12, 1.e-3],
#         fig_name='TS'+str(i+1),
#         legend_loc='upper right'
#     )

## StS vs CG vs Park
Setting up

In [346]:
if FLAGS['sts_cg_park']:
    path_ = path + '/sts_cg_park/'
    if not os.path.exists(path_):
        os.makedirs(path_)

case = 2

# Initialize conditions
if case == 1:
    T = 10000.
    T0 = 2000.
    P0 = 3000.
    XO2 = 0.8
else:
    T = 15000.
    T0 = 1000.
    P0 = 1000.
    XO2 = 0.95

# QSS distribution time instant
if case == 1:
    t_QSS = 2.5e-6
else:
    t_QSS = 1.6e-6

# Paths to solutions
# Park
parkQSS_file = cwd + f'/src/sts_cg_park.{case}/park/QSS/output_box/box.dat'
park2001_file = cwd + f'/src/sts_cg_park.{case}/park/2001/output_box/box.dat'
# # CG
# RVE1_file = cwd + f'/src/sts_cg_park.{case}/cg/RVE1/outputs/out.csv'
# RVE27_file = cwd + f'/src/sts_cg_park.{case}/cg/RVE27/outputs/out.csv'
# StS
sts_file = cwd + f'/src/sts_cg_park.{case}/sts/excit_diss/output_box/box.dat'
pop_file = cwd + f'/src/sts_cg_park.{case}/sts/excit_diss/output_box/pop_O2.dat'

# Labels
lbl = {
    'RVE1': 'CG-RVE1',
    'RVE27': 'CG-RVE27',
    'park': 'Park',
    'parkQSS': 'Park-QSS',
    'park2001': 'Park-2001',
    'sts': 'StS'
}

# Define specie thermodynamics
spTh = speciesThermos[-1]['O2']

# Park's energy model
RR_HO = False

Generating Initial Conditions

In [347]:
if FLAGS['sts_cg_park']:
    # Generating Initial Conditions
    rho_s = np.zeros((1,dataGen.num_bins+1), dtype=np.float64)
    XO = 1. - XO2
    nd = P0 / (const.UKB * T0)
    Q_bins = dataGen.partition_func(T0)
    rho_s[0,:-1] = XO2 * Q_bins / sum(Q_bins) * 2.0 * const.mass['O'] * nd
    rho_s[0, -1] = XO  * const.mass['O'] * nd
    rho = np.sum(rho_s, axis=1, keepdims=True)
    Ttr = utils.reshape_to_2DArray(T)
    ic = np.hstack((Ttr, rho, np.divide(rho_s, rho)))

Get levels properties and populations

In [348]:
# Properties
E_lev = np.array(spTh.specie.rv_lev['E']).squeeze()/const.EV_to_J # [eV]
g_lev = np.array(spTh.specie.rv_lev['g']).squeeze()
n_lev = len(spTh.specie.rv_lev['g'])

# States number densities
def n_states(instants, file):
    with open(file, "r") as f:
        lines = f.readlines()
        del lines[:1]
    states = []
    for i in tqdm(range(instants)):
        states.append(np.loadtxt(lines[i*(n_lev+1)+1:(n_lev+1)*(i+1)], dtype=float).T[1].reshape(1,-1))
    return np.vstack(states)*g_lev

Energy evaluation

In [349]:
# Coarse-Graining
def E_cg(rho, T, Y, spTh=speciesThermos[-1]['O2']):
    # Update thermos
    spTh.update(utils.reshape_to_2DArray(T))
    # Relative mole fraction
    n_i = Y * rho / spTh.thermo.specie.M * const.UNA
    X_rel = n_i/np.sum(n_i, axis=1, keepdims=True)
    # Evaluate energy
    e_int = spTh.thermo.e_int/const.EV_to_J*spTh.specie.m*X_rel
    return np.sum(e_int, axis=1, keepdims=True)

# Park
def E_park(rho, Tr, Tv, spTh=speciesThermos[-1]['O2']):
    if RR_HO:
        # Rotational energy
        E_r = spTh.specie.R*Tr/const.EV_to_J*spTh.specie.m
        # Vibrational energy
        r = spTh.specie.theta_v/Tv
        E_v = spTh.specie.R*Tv*r/(np.exp(r)-1.)/const.EV_to_J*spTh.specie.m
        # Total internal energy
        e_int = E_r+E_v
    else:
        spTh = speciesThermos[0]['O2']
        # Update thermos
        spTh.update(utils.reshape_to_2DArray(Tv))
        # Evaluate energy
        e_int = spTh.thermo.e_int/const.EV_to_J*spTh.specie.m
    return np.array(e_int)

# State-to-State
def E_sts(rho, X_rel):
    return np.sum(X_rel * E_lev, axis=1)

Evaluate mass fraction and internal energy

In [350]:
def extract_data(df):
    t = df[[col for col in df.columns if 't' in col]].to_numpy()
    X_O = df[[col for col in df.columns if col == 'X_O']].to_numpy()
    X_O2 = df[[col for col in df.columns if col == 'X_O2']].to_numpy()
    return t, X_O, X_O2

def Y_O2(X_O, X_O2):
    return 2*X_O2/(X_O + 2*X_O2)

if FLAGS['sts_cg_park']:
    
    data, t, Y, E = {}, {}, {}, {}
    
#     # Solution from CG-RVE1
#     df = pd.read_csv(RVE1_file)
#     data['RVE1'] = df
#     Y_ = df[[col for col in df.columns if 'Y_O2' in col]].to_numpy()
#     t['RVE1'] = df[[col for col in df.columns if 't' in col]].to_numpy()
#     Y['RVE1'] = np.sum(Y_, axis=1)
#     E['RVE1'] = E_cg(rho, T, Y_, spTh=speciesThermos[0]['O2']).squeeze()
    
#     # Solution from CG-RVE27
#     df = pd.read_csv(RVE27_file)
#     data['RVE27'] = df
#     Y_ = df[[col for col in df.columns if 'Y_O2' in col]].to_numpy()
#     t['RVE27'] = df[[col for col in df.columns if 't' in col]].to_numpy()
#     Y['RVE27'] = np.sum(Y_, axis=1)
#     E['RVE27'] = E_cg(rho, T, Y_).squeeze()
    
    # Solution from CG-RVE27
    data['RVE27'] = dataGen.testing_data(ic_test=ic, save=False)[0]
    t['RVE27'] = data['RVE27'][0][1]
    Y['RVE27'] = data['RVE27'][1][0][:,0]
    E['RVE27'] = E_cg(rho, T, data['RVE27'][1][-1]).squeeze()
    
    # Solution from Park 2001
    data['park2001'] = pd.read_csv(
        park2001_file, sep="  ", header=1, engine='python'
    )
    Tv = data['park2001'][
        [col for col in data['park2001'].columns if 'Tv' in col]
    ].to_numpy()
    t['park2001'], X_O, X_O2 = extract_data(data['park2001'])
    Y['park2001'] = Y_O2(X_O, X_O2)
    E['park2001'] = E_park(rho, Tv, Tv).squeeze()
    
    # Solution from Park QSS
    data['parkQSS'] = pd.read_csv(
        parkQSS_file, sep="  ", header=1, engine='python'
    )
    Tv = data['parkQSS'][
        [col for col in data['parkQSS'].columns if 'Tv' in col]
    ].to_numpy()
    t['parkQSS'], X_O, X_O2 = extract_data(data['parkQSS'])
    Y['parkQSS'] = Y_O2(X_O, X_O2)
    E['parkQSS'] = E_park(rho, Tv, Tv).squeeze()
    
    # Solution from StS
    data['sts'] = pd.read_csv(sts_file, sep="  ", header=1, engine='python')
    t['sts'], X_O, X_O2 = extract_data(data['sts'])
    Y['sts'] = Y_O2(X_O, X_O2)
    n_sts = n_states(instants=len(Y['sts']), file=pop_file)
    X_rel = n_sts/np.sum(n_sts, axis=1, keepdims=True)
    E['sts'] = E_sts(rho, X_rel).squeeze()
    

Plot mass fractions and internal energy

In [351]:
def plot_var(
    t,
    y,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    xlim=None,
    ylim=None,
    fig_path=None
):
    fig, ax = plt.subplots()

    # Set up figure -----------------------------------------------------------
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if x_scale is not None:
        ax.set_yscale(y_scale)
    if ylim is not None:
        ax.set_ylim(ylim)

    # Plot figure -------------------------------------------------------------
    style = dict(
        marker='o',
        fillstyle='full',
        markersize=3,
        lw=1.
    )
    # Loop over features
    ax.plot(
        t['sts'],
        y['sts'],
        linestyle='--',
        label=lbl['sts'],
        lw=1.,
        c='k'
    )
#     for i, k in enumerate(['RVE1', 'RVE27', 'parkQSS', 'park2001']):
    for i, k in enumerate(['RVE27', 'parkQSS', 'park2001']):
        ax.plot(
            t[k],
            y[k],
            label=lbl[k],
            lw=1.
        )
    # Add legend
    ax.legend(
        fontsize=14,
        loc='best'
    )

    # Save figure -------------------------------------------------------------
    fig.savefig(fig_path)


In [352]:
if FLAGS['sts_cg_park']:
    plot_var(
        t,
        Y,
        labels=[r'$t$ [s]', r'$Y$'],
        scales=['log', 'linear'],
        xlim=[1.e-8, 1.e-5],
        fig_path=path_ + '/mass_log.pdf'
    )
    
    plot_var(
        t,
        E,
        labels=[r'$t$ [s]', r'$E$ [eV]'],
        scales=['log', 'linear'],
        xlim=[1.e-10, 1.e-4],
        fig_path=path_ + '/energy.pdf'
    )

### QSS states distribution

In [353]:
def plot_levels_dist(
    x,
    y,
    labels=[r'$\epsilon_i$ [eV]', r'$n_i/g_i$ [m$^{-3}$]'],
    scales=['linear', 'log'],
    label=None,
    color=None,
    fig_path=None,
    fig_ax=None,
    save=True
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if x_scale is not None:
        ax.set_yscale(y_scale)

    # Plot figure -------------------------------------------------------------
    style = dict(
        lw=1.,
        marker='o',
        fillstyle='full',
        markersize=0.3,
        linestyle='none'
    )
    if color is not None:
        style['c'] = color
    if label is not None:
        style['label'] = label
    # Loop over features
    ax.plot(
        x,
        y,
        **style
    )

    # Save figure -------------------------------------------------------------
    if save:
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax
    

In [354]:
if FLAGS['sts_cg_park']:
    # Get index for QSS
    idx = tf.nest.map_structure(
        lambda x: np.argmin(np.absolute(x-t_QSS), axis=0), t
    )

    # Number densities
    n = {}

    # States population from StS --------------------------------------------------
    n['sts'] = n_sts[idx['sts']].squeeze()

    # States population from CG (Timescale 1) -------------------------------------
    composition = {
        'O2': data['RVE27'][1][-1][idx['RVE27']].reshape(1, -1),
        'O': np.take(data['RVE27'][1][0], [1], axis=1)[idx['RVE27']].reshape(1, -1)
    }
    n['RVE27'] = np.concatenate(
        lev_dist(ts_mixtures[-1], Ttr, rho, composition)[0], axis=1
    ).squeeze()

    # States population from Park -------------------------------------------------
    df, i = data['parkQSS'], idx['parkQSS']
    # Get physical quantites
    Tv = df[[col for col in df.columns if 'Tv' in col]].to_numpy()[i]
    n_QSS = df[[col for col in df.columns if 'nb' in col]].to_numpy()[i]
    X_O2 = df[[col for col in df.columns if col == 'X_O2']].to_numpy()[i]
    n_O2 = n_QSS * X_O2
    # Get partition functions
    q = g_lev*np.exp(-E_lev*const.EV_to_J/(const.UKB*Tv))
    n['parkQSS'] = (n_O2 * q / np.sum(q)).squeeze()

Normalized QSS states distribution

In [355]:
if FLAGS['sts_cg_park']:
    
    # Get normalized states distribution
    n_norm = tf.nest.map_structure(lambda x: x/g_lev/np.sum(x), n)

    # Plot normalized states distribution
    fig, ax = plt.subplots()
    labels = []
    for name, value in n_norm.items():
        if name == 'sts':
            color = 'grey'
        else:
            color = None
        fig, ax = plot_levels_dist(
            x=E_lev,
            y=value,
            labels=[r'$\epsilon_i$ [eV]', r'$n_i/(n_{_{\text{O}_2}}\,\,g_i)$'],
            color=color,
            fig_ax=(fig, ax),
            save=False
        )
        labels.append(lbl[name])

    # Add legend
    colors = [l.get_color() for l in plt.gca().lines]
    ax.legend(
        [plt.plot([], [], c=c)[0] for c in colors],
        labels,
        fontsize=14,
        loc='best'
    )

    # Save figure
    fig.savefig(path_ + '/lev_dist_norm.pdf')
    
    # ----------------------------------------------------------------------- #
    
    # Get normalized states distribution
    n_norm = tf.nest.map_structure(lambda x: x/g_lev, n)

    # Plot normalized states distribution
    fig, ax = plt.subplots()
    labels = []
    for name, value in n_norm.items():
        if name == 'sts':
            color = 'grey'
        else:
            color = None
        fig, ax = plot_levels_dist(
            x=E_lev,
            y=value,
            color=color,
            fig_ax=(fig, ax),
            save=False
        )
        labels.append(lbl[name])

    # Add legend
    colors = [l.get_color() for l in plt.gca().lines]
    ax.legend(
        [plt.plot([], [], c=c)[0] for c in colors],
        labels,
        fontsize=14,
        loc='best'
    )

    # Save figure
    fig.savefig(path_ + '/lev_dist.pdf')
    

QSS states distribution

In [356]:
if FLAGS['sts_cg_park']:
    # Get states distribution
    n = tf.nest.map_structure(lambda x: x/g_lev, n)

    # Plot states distribution for each model
    it = 0
    for name, value in n.items():
        if name == 'sts':
            color = 'k'
        else:
            color = COLORS[it]
            it += 1
        plot_levels_dist(
            x=E_lev,
            y=value,
            label=None,
            color=color,
            fig_path=path_ + f'/lev_dist_{lbl[name]}.pdf'
        )